<a href="https://colab.research.google.com/github/KingsWorkflow/KtmAir-Quality-and-Weather-ETL-Pipeline/blob/main/DataAnalaysisFInalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kathmandu Valley Air Quality and Weather ETL Pipeline: Understanding Pollution Patterns for Public Health Awareness**

**Setup**




In [8]:
import json, time, hashlib, requests
import pandas as pd
from pathlib import Path
from datetime import datetime

PROJECT = Path("/content/kathmandu_aq_etl")
RAW = PROJECT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Optional: keep files after the Colab session ends
# from google.colab import drive; drive.mount('/content/drive')
# PROJECT = Path("/content/drive/MyDrive/kathmandu_aq_etl")

print("Project folder:", PROJECT)

Project folder: /content/kathmandu_aq_etl


**API configuration and a safe request helper**

In [9]:
from google.colab import userdata

API_KEY = userdata.get('OPENAQ')
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"

def api_get(path, params=None, retries=5):
    """GET with retry/backoff for rate limits (HTTP 429) and temporary errors."""
    url = f"{BASE_URL}{path}"
    for attempt in range(retries):
        r = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            wait = 20 * (attempt + 1)
            print(f"  HTTP {r.status_code}, retrying in {wait}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
    raise RuntimeError(f"Failed after {retries} tries: {url}")

# Quick connectivity test
print(api_get("/countries", {"limit": 1})["meta"])

{'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 1, 'found': 159}


**Discover monitoring stations in the Kathmandu Valley**

In [10]:
# Bounding box: min_lon, min_lat, max_lon, max_lat (covers Kathmandu, Lalitpur, Bhaktapur)
KTM_BBOX = "85.15,27.55,85.60,27.85"

resp = api_get("/locations", {"bbox": KTM_BBOX, "limit": 1000})
locations = resp["results"]
print("Stations found:", len(locations))

# Preserve the raw API response exactly as received
with open(RAW / "openaq_locations_raw.json", "w") as f:
    json.dump(resp, f, indent=2)

station_rows, sensor_rows = [], []
for loc in locations:
    coords = loc.get("coordinates") or {}
    station_rows.append({
        "location_id": loc["id"],
        "name": loc.get("name"),
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
        "provider": (loc.get("provider") or {}).get("name"),
        "owner": (loc.get("owner") or {}).get("name"),
        "is_mobile": loc.get("isMobile"),
        "first_utc": (loc.get("datetimeFirst") or {}).get("utc"),
        "last_utc": (loc.get("datetimeLast") or {}).get("utc"),
    })
    for s in loc.get("sensors", []):
        p = s.get("parameter") or {}
        sensor_rows.append({
            "sensor_id": s["id"],
            "location_id": loc["id"],
            "parameter": p.get("name"),
            "units": p.get("units"),
        })

stations_raw = pd.DataFrame(station_rows)
sensors_raw = pd.DataFrame(sensor_rows)
stations_raw.to_csv(RAW / "stations_raw.csv", index=False)
sensors_raw.to_csv(RAW / "sensors_raw.csv", index=False)

display(stations_raw)
display(sensors_raw.groupby(["location_id", "parameter"]).size().unstack(fill_value=0))

Stations found: 60


,location_id,name,latitude,longitude,provider,owner,is_mobile,first_utc,last_utc
0,3459,Embassy Kathmandu,27.738703,85.336206,AirNow,Unknown Governmental Organization,False,2017-03-03T00:00:00Z,2026-06-17T23:15:00Z
1,3460,Phora Durbar Kathman,27.712464,85.315703,AirNow,Unknown Governmental Organization,False,2017-03-03T00:00:00Z,2025-03-24T14:15:00Z
2,1236017,"Dhathutole, Handigaun",27.727502,85.330135,AirGradient,Unknown Governmental Organization,False,2023-07-15T13:00:00Z,2026-06-01T04:00:00Z
3,2848864,"Dabali, Handigaun",27.681891,85.287078,AirGradient,Unknown Person,False,2024-05-16T03:00:00Z,2026-10-09T02:00:00Z
4,4121681,Kathmandu Institute of Applied Sciences (KIAS),27.671693,85.301670,AirGradient,Sapana Bhujel,False,2025-05-01T09:00:00Z,2026-10-09T02:00:00Z
5,5506835,Gaushala Chowk (SC-01) - GD Labs,27.707763,85.343189,AirGradient,Green Decision Labs and Research,False,2025-09-19T00:00:00Z,2026-10-09T02:00:00Z
6,5509787,Baluwatar (SC-02) - GD Labs,27.724944,85.331062,AirGradient,Green Decision Labs and Research,False,2025-09-25T07:00:00Z,2026-10-09T02:00:00Z
7,5509788,Lagankhel (SC - 05) - GD Labs,27.666459,85.323093,AirGradient,Green Decision Labs and Research,False,2025-10-03T00:00:00Z,2026-10-09T02:00:00Z
8,5518333,Sifal(SC-03)- GD Labs,27.710794,85.340591,AirGradient,Green Decision Labs and Research,False,2025-12-24T08:00:00Z,2026-10-09T02:00:00Z
9,5519358,Lamtangil (SC-04)- GD Labs,27.731368,85.336783,AirGradient,Green Decision Labs and Research,False,2025-12-24T09:00:00Z,2026-10-09T02:00:00Z


parameter,o3,pm1,pm10,pm25,relativehumidity,temperature,um003
location_id,,,,,,,
3459,1,0,0,1,0,0,0
3460,1,0,0,1,0,0,0
1236017,0,1,1,1,1,1,1
2848864,0,1,1,1,1,1,1
4121681,0,1,0,1,1,1,1
5506835,0,1,0,1,1,1,1
5509787,0,1,0,1,1,1,1
5509788,0,1,0,1,1,1,1
5518333,0,1,0,1,1,1,1


**Choose sensors (pollutants, active since 2023)**

In [11]:
TARGET_PARAMS = ["pm25", "pm10", "no2", "o3", "co", "so2"]

st = stations_raw.copy()
st["last_utc"] = pd.to_datetime(st["last_utc"], utc=True, errors="coerce")
st["first_utc"] = pd.to_datetime(st["first_utc"], utc=True, errors="coerce")

# Keep stations that have data overlapping our study window
active = st[(st["last_utc"] >= "2023-01-01") & (st["first_utc"] <= "2025-12-31")]
selected = sensors_raw[
    sensors_raw["location_id"].isin(active["location_id"])
    & sensors_raw["parameter"].isin(TARGET_PARAMS)
].merge(active[["location_id", "name"]], on="location_id")

print("Selected sensors:", len(selected))
display(selected)

Selected sensors: 55


,sensor_id,location_id,parameter,units,name
0,7713,3459,o3,ppm,Embassy Kathmandu
1,7710,3459,pm25,µg/m³,Embassy Kathmandu
2,7712,3460,o3,ppm,Phora Durbar Kathman
3,7711,3460,pm25,µg/m³,Phora Durbar Kathman
4,8462426,1236017,pm10,µg/m³,"Dhathutole, Handigaun"
5,6530219,1236017,pm25,µg/m³,"Dhathutole, Handigaun"
6,9235741,2848864,pm10,µg/m³,"Dabali, Handigaun"
7,9235746,2848864,pm25,µg/m³,"Dabali, Handigaun"
8,12914045,4121681,pm25,µg/m³,Kathmandu Institute of Applied Sciences (KIAS)
9,14153358,5506835,pm25,µg/m³,Gaushala Chowk (SC-01) - GD Labs


**Testing on 7 days first, then running the full pull**

In [12]:
def fetch_sensor_hours(sensor_id, dt_from, dt_to):
    """Hourly-aggregated measurements for one sensor, with pagination."""
    rows, page = [], 1
    while True:
        j = api_get(f"/sensors/{sensor_id}/hours", {
            "datetime_from": dt_from, "datetime_to": dt_to,
            "limit": 1000, "page": page,
        })
        batch = j["results"]
        rows.extend(batch)
        if len(batch) < 1000:
            break
        page += 1
        time.sleep(1.1)          # stay under the rate limit
    return rows

def pull(selected_sensors, dt_from, dt_to):
    frames = []
    for _, s in selected_sensors.iterrows():
        data = fetch_sensor_hours(int(s.sensor_id), dt_from, dt_to)
        print(f"sensor {s.sensor_id} | {s['name']} | {s.parameter}: {len(data)} rows")
        if data:
            df = pd.json_normalize(data)
            df["sensor_id"] = s.sensor_id
            df["location_id"] = s.location_id
            df["parameter_name"] = s.parameter
            frames.append(df)
        time.sleep(1.1)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# --- TEST RUN: 7 days only ---
test = pull(selected, "2025-01-01T00:00:00Z", "2025-01-08T00:00:00Z")
print(test.shape)
display(test.head())

sensor 7713 | Embassy Kathmandu | o3: 168 rows
sensor 7710 | Embassy Kathmandu | pm25: 168 rows
sensor 7712 | Phora Durbar Kathman | o3: 168 rows
sensor 7711 | Phora Durbar Kathman | pm25: 168 rows
sensor 8462426 | Dhathutole, Handigaun | pm10: 155 rows
sensor 6530219 | Dhathutole, Handigaun | pm25: 155 rows
sensor 9235741 | Dabali, Handigaun | pm10: 155 rows
sensor 9235746 | Dabali, Handigaun | pm25: 155 rows
sensor 12914045 | Kathmandu Institute of Applied Sciences (KIAS) | pm25: 0 rows
sensor 14153358 | Gaushala Chowk (SC-01) - GD Labs | pm25: 0 rows
sensor 14207425 | Baluwatar (SC-02) - GD Labs | pm25: 0 rows
sensor 14272754 | Lagankhel (SC - 05) - GD Labs | pm25: 0 rows
sensor 14975414 | Sifal(SC-03)- GD Labs | pm25: 0 rows
sensor 14976381 | Lamtangil (SC-04)- GD Labs | pm25: 0 rows
sensor 14153468 | Pulchowk (SC-44) - GD Labs | pm25: 0 rows
sensor 14153373 | Nakhipot (SC-08) - GD Labs | pm25: 0 rows
sensor 14153363 | Sunakothi (SC - 06) - GD Labs | pm25: 0 rows
sensor 14153368 | 

/tmp/ipykernel_2101/979750207.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


,value,coordinates,flagInfo.hasFlags,parameter.id,parameter.name,parameter.units,parameter.displayName,period.label,period.interval,period.datetimeFrom.utc,...,coverage.observedInterval,coverage.percentComplete,coverage.percentCoverage,coverage.datetimeFrom.utc,coverage.datetimeFrom.local,coverage.datetimeTo.utc,coverage.datetimeTo.local,sensor_id,location_id,parameter_name
0,0.002,None,True,10,o3,ppm,None,1hour,01:00:00,2024-12-31T23:15:00Z,...,01:00:00,100.0,100.0,2024-12-31T23:15:00Z,2025-01-01T05:00:00+05:45,2025-01-01T00:15:00Z,2025-01-01T06:00:00+05:45,7713,3459,o3
1,0.002,None,False,10,o3,ppm,None,1hour,01:00:00,2025-01-01T00:15:00Z,...,01:00:00,100.0,100.0,2025-01-01T00:15:00Z,2025-01-01T06:00:00+05:45,2025-01-01T01:15:00Z,2025-01-01T07:00:00+05:45,7713,3459,o3
2,0.003,None,False,10,o3,ppm,None,1hour,01:00:00,2025-01-01T01:15:00Z,...,01:00:00,100.0,100.0,2025-01-01T01:15:00Z,2025-01-01T07:00:00+05:45,2025-01-01T02:15:00Z,2025-01-01T08:00:00+05:45,7713,3459,o3
3,0.004,None,False,10,o3,ppm,None,1hour,01:00:00,2025-01-01T02:15:00Z,...,01:00:00,100.0,100.0,2025-01-01T02:15:00Z,2025-01-01T08:00:00+05:45,2025-01-01T03:15:00Z,2025-01-01T09:00:00+05:45,7713,3459,o3
4,0.010,None,False,10,o3,ppm,None,1hour,01:00:00,2025-01-01T03:15:00Z,...,01:00:00,100.0,100.0,2025-01-01T03:15:00Z,2025-01-01T09:00:00+05:45,2025-01-01T04:15:00Z,2025-01-01T10:00:00+05:45,7713,3459,o3
